# FontDiffusion fd_cache generator (Colab GPU)

Generate `fd_cache/` images for all books in one batch on Colab.

**Inputs (uploaded from Mac):**
- `colab_diffusion/exports/chars_<book>.txt` — chars to generate per book
- `colab_diffusion/exports/style_<book>.png` — style reference image per book
- `colab_diffusion/exports/MANIFEST.json` — orchestration
- `font_diffusion/` — model code + checkpoints
- `dict/` — dictionaries (only needed if rerunning aggregation here)

**Output:** one `fd_cache_<book>.zip` per book containing `U+XXXX.png` files.
Download these zips and unzip into `prepared/<book>/fd_cache/` on Mac.

**Required runtime:** GPU (T4 or better). Diffusion inference is slow on CPU.

## 1. Install deps + check GPU

In [ ]:
!pip install -q diffusers accelerate einops kornia info-nce-pytorch fonttools pygame scikit-image safetensors
!nvidia-smi

## 2. Mount drive + locate project root

Adjust `PROJECT_ROOT` to where you uploaded the repo (e.g. via `git clone` or Google Drive).

In [ ]:
import os, sys, json, shutil, zipfile
from pathlib import Path

# EDIT THIS: path to the GanNhanOCR project root on Colab
PROJECT_ROOT = Path('/content/GanNhanOCR')
EXPORTS = PROJECT_ROOT / 'colab_diffusion' / 'exports'

assert EXPORTS.exists(), f'Missing {EXPORTS}. Upload colab_diffusion/exports/ first.'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Working dir:', os.getcwd())

with open(EXPORTS / 'MANIFEST.json') as f:
    manifest = json.load(f)
print(f'Books to process: {list(manifest["books"].keys())}')
print(f'Total unique chars (union): {manifest["union_chars"]}')

## 3. Load FontDiffusion generator (once, reused for all books)

In [ ]:
from core.ranking.fontdiffusion_gen import FontDiffusionGenerator

ckpt_dir = manifest['fontdiffusion_ckpt_subpath']
phase1_dir = manifest['fontdiffusion_phase1_ckpt_subpath']
font_path = manifest['font_path_subpath']

for p in (ckpt_dir, phase1_dir, font_path):
    assert Path(p).exists(), f'Missing: {p}'

print('Loading FontDiffusion (one-time)...')
# We re-use one generator across books; cache_dir is reset per book below
generator = FontDiffusionGenerator(
    ckpt_dir=ckpt_dir,
    phase1_ckpt_dir=phase1_dir,
    font_path=font_path,
    cache_dir='/tmp/fd_global',  # placeholder
)
print('Loaded.')

## 4. Generate per book and zip output

In [ ]:
OUT_ZIP_DIR = PROJECT_ROOT / 'colab_diffusion' / 'fd_cache_zips'
OUT_ZIP_DIR.mkdir(parents=True, exist_ok=True)

for book, info in manifest['books'].items():
    if not info.get('chars_file') or not info.get('style_image'):
        print(f'  [skip] {book}: missing chars or style image')
        continue

    chars_file = EXPORTS / info['chars_file']
    style_image = str(EXPORTS / info['style_image'])
    chars = [line.strip() for line in open(chars_file, encoding='utf-8') if line.strip()]

    cache_dir = PROJECT_ROOT / 'colab_diffusion' / f'_tmp_{book}'
    cache_dir.mkdir(parents=True, exist_ok=True)
    generator.cache_dir = str(cache_dir)

    print(f'\n>>> {book}: generating {len(chars)} chars...')
    generated = generator.generate(chars, style_image, style_name=book)
    print(f'    Generated: {len(generated)}/{len(chars)}')

    # Zip the cache_dir
    zip_path = OUT_ZIP_DIR / f'fd_cache_{book}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for png in cache_dir.rglob('U+*.png'):
            z.write(png, arcname=png.name)
    print(f'    Wrote {zip_path}')

## 5. Download zips to local Mac

After download, on Mac:

```sh
for book in CacThanhTruyen2 CacThanhTruyen4 CacThanhTruyen11 \
            SachThanhTruyen2 SachThanhTruyen4 SachThanhTruyen11; do
  mkdir -p "prepared/$book/fd_cache"
  unzip -o "fd_cache_${book}.zip" -d "prepared/$book/fd_cache/"
done
```

In [ ]:
from google.colab import files
for z in sorted(OUT_ZIP_DIR.glob('fd_cache_*.zip')):
    print(f'Downloading {z.name} ({z.stat().st_size/1024/1024:.1f} MB)...')
    files.download(str(z))